# 01 · Análise exploratória do dataset sintético

Objetivo: verificar se o dataset é plausível e entender como as fraudes aparecem nos dados antes de modelar.

Antes, gere os dados com `python scripts/gerar_dataset.py` (na raiz do repositório).

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

from pixfraud.anomaly import prepare_matrix
from pixfraud.features import build_features
from pixfraud.graph import build_graph
from pixfraud.simulation import load_dataset

accounts, tx = load_dataset("../data/synthetic")
tx.head()

## Visão geral

In [ ]:
print(accounts["role"].value_counts(), end="\n\n")
tx["scenario"].value_counts()

## Valores

Os golpes devem ter valores maiores que o típico (parâmetro `fraud_amount_shift`).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.logspace(0, np.log10(tx["amount"].max()), 50)
for is_fraud, group in tx.groupby("is_fraud"):
    ax.hist(group["amount"], bins=bins, alpha=0.6, density=True, label="fraude" if is_fraud else "normal")
ax.set_xscale("log")
ax.set_xlabel("valor (R$)")
ax.legend();

## Horário

No gerador atual os golpes têm horário uniforme, enquanto os Pix legítimos seguem `HOUR_WEIGHTS` (ver *Limitações conhecidas* em `data/README.md`).

In [ ]:
hourly = pd.crosstab(tx["timestamp"].dt.hour, tx["is_fraud"], normalize="columns")
hourly.columns = ["normal", "fraude"]
hourly.plot(kind="bar", figsize=(10, 4), xlabel="hora", ylabel="fração das transações");

## Estrutura da rede

In [ ]:
G = build_graph(tx, accounts["account_id"])
print(G)
print("densidade:", nx.density(G))
print("componentes fracamente conexas:", nx.number_weakly_connected_components(G))

counts = pd.Series(dict(G.degree())).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(counts.index, counts.values, "o", markersize=3)
ax.set_xlabel("grau")
ax.set_ylabel("nº de contas");

## Atributos por papel

Mediana de cada atributo por papel da conta: quais atributos separam as fraudes das contas legítimas?

In [ ]:
features = build_features(tx, accounts, G)
roles = accounts.set_index("account_id")["role"]
features.groupby(roles).median().T.round(4)

## Escolha do `eps` do DBSCAN

Gráfico de k-distance: distância de cada conta ao `min_samples`-ésimo vizinho, em ordem crescente. O "cotovelo" da curva é um bom candidato a `eps`.

In [ ]:
min_samples = 10
X = prepare_matrix(features)
distances, _ = NearestNeighbors(n_neighbors=min_samples).fit(X).kneighbors(X)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.sort(distances[:, -1]))
ax.set_xlabel("contas (ordenadas)")
ax.set_ylabel(f"distância ao {min_samples}º vizinho");

## Próximos passos

- Comparar valores e horários com as estatísticas do BCB: `python scripts/baixar_dados_bcb.py transacoes 202506 --filtro "AnoMes eq 202506"`.
- Ajustar `configs/experimento.yaml` e registrar as decisões.